#Projeto Final - Big Data e Engenharia de Dados
- Lavínia Dantas de Mesquita (Lava) - 2025022536
- Daniel Lins Silva Andrade (Baiano) - 2019010787
- Luis Felipe Pereira de Paiva Pordeus - 2024012641

## Setup da Arquitetura Medallion
- Criação dos esquemas onde as tabelas do dataset serão salvas, utilizando a organização da Arquitetura Medallion

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dc_catalog; -- Catálogo do Lakehouse
CREATE SCHEMA IF NOT EXISTS dc_catalog.bronze; -- Camada Bronze
CREATE VOLUME IF NOT EXISTS dc_catalog.bronze.landingzone; -- Volume para armazenar os arquivos logo após a ingestão
CREATE SCHEMA IF NOT EXISTS dc_catalog.silver; -- Camada Silver
CREATE SCHEMA IF NOT EXISTS dc_catalog.gold; -- Camada Gold
--USE CATALOG dc_catalog;
--USE SCHEMA dc_schema;

# Camada Bronze

## Ingestão dos Dados na Landing Zone
- Utilizamos o kagglehub para carregar os dados diretamente do site que contém os dados a serem utlizados.

In [0]:
# Imports
from pyspark.sql import functions as F # Funções para manipulação de colunas, operações matemáticas, agregações e condições
from pyspark.sql.types import DoubleType, LongType # Tipos do spark que serão bastante usados
from pyspark.sql import DataFrame # Dataframes
from pyspark.sql.functions import monotonically_increasing_id # Para criar coluna de id da camada gold (esquema estrela)

import pandas as pd # Manipulação de dados	
import numpy as np # Base matemática para álgebra linear e operações

from sklearn.model_selection import train_test_split # Divisão de dados em treino e teste para machine learning
from sklearn.preprocessing import StandardScaler # Padronização dos dados com média 0 e desvio 1
from sklearn.metrics import (
    classification_report, # Classificação
    confusion_matrix, # Matriz de Confusão
    mean_absolute_error, # Regressão
    mean_squared_error, # Penalização
    r2_score, # Ajuste de de qualidade
    accuracy_score # Acurácia
)
from sklearn.neural_network import MLPClassifier, MLPRegressor # Redes densas classificadora e regressora

import matplotlib.pyplot as plt #Plot de gráficos
import seaborn as sns # Plot de gráficos

In [0]:
!pip install kagglehub # Kaggle
import kagglehub
import os
import pandas as pd

volume_path = "/Volumes/dc_catalog/bronze/landingzone" # Caminho da Landing Zone
dbutils.fs.mkdirs(volume_path) # Garante que a pasta existe

# Baixa o dataset do Kaggle
dataset_path = kagglehub.dataset_download("nosbielcs/brazilian-delivery-center")
print("Dataset baixado em:", dataset_path)
files = os.listdir(dataset_path) # Lista arquivos baixados

for file in files:
    local_file_path = os.path.join(dataset_path, file)
    destination_path = f"{volume_path}/{file}"

    if file.endswith(".csv"): # Se o arquivo existir, leiam crie o dataframe e salve o arquivo no local correto
        pdf = pd.read_csv(
            local_file_path,
            encoding="latin1"
        )
        df = spark.createDataFrame(pdf)
        df.write.mode("overwrite").csv(destination_path)
        print(f"Arquivo salvo: {destination_path}")

print("\nIngestão concluída!")

### Esquema de dados brutos

In [0]:
# Esse esquema segue o formato original da base de dados
from pyspark.sql.types import *

stores_schema = StructType([
    StructField("store_id", IntegerType(), True),
    StructField("hub_id", IntegerType(), True),
    StructField("store_name", StringType(), True),
    StructField("store_segment", StringType(), True),
    StructField("store_plan_price", DoubleType(), True),
    StructField("store_latitude", DoubleType(), True),
    StructField("store_longitude", DoubleType(), True)
])

payments_schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("payment_order_id", IntegerType(), True),
    StructField("payment_amount", DoubleType(), True),
    StructField("payment_fee", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("payment_status", StringType(), True)
])

orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("channel_id", IntegerType(), True),
    StructField("payment_order_id", IntegerType(), True),
    StructField("delivery_order_id", IntegerType(), True),

    StructField("order_status", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("order_delivery_fee", DoubleType(), True),
    StructField("order_delivery_cost", DoubleType(), True),

    StructField("order_created_hour", IntegerType(), True),
    StructField("order_created_minute", IntegerType(), True),
    StructField("order_created_day", IntegerType(), True),
    StructField("order_created_month", IntegerType(), True),
    StructField("order_created_year", IntegerType(), True),

    # Timestamps de processo
    StructField("order_moment_created", TimestampType(), True),
    StructField("order_moment_accepted", TimestampType(), True),
    StructField("order_moment_ready", TimestampType(), True),
    StructField("order_moment_collected", TimestampType(), True),
    StructField("order_moment_in_expedition", TimestampType(), True),
    StructField("order_moment_delivering", TimestampType(), True),
    StructField("order_moment_delivered", TimestampType(), True),
    StructField("order_moment_finished", TimestampType(), True),

    # Métricas de tempo
    StructField("order_metric_collected_time", DoubleType(), True),
    StructField("order_metric_paused_time", DoubleType(), True),
    StructField("order_metric_production_time", DoubleType(), True),
    StructField("order_metric_walking_time", DoubleType(), True),
    StructField("order_metric_expediton_speed_time", DoubleType(), True),
    StructField("order_metric_transit_time", DoubleType(), True),
    StructField("order_metric_cycle_time", DoubleType(), True)
])

hubs_schema = StructType([
    StructField("hub_id", IntegerType(), True),
    StructField("hub_name", StringType(), True),
    StructField("hub_city", StringType(), True),
    StructField("hub_state", StringType(), True),
    StructField("hub_latitude", DoubleType(), True),
    StructField("hub_longitude", DoubleType(), True)
])

drivers_schema = StructType([
    StructField("driver_id", IntegerType(), True),
    StructField("driver_modal", StringType(), True),
    StructField("driver_type", StringType(), True)
])

deliveries_schema = StructType([
    StructField("delivery_id", IntegerType(), True),
    StructField("delivery_order_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("delivery_distance_meters", DoubleType(), True),
    StructField("delivery_status", StringType(), True)
])

channels_schema = StructType([
    StructField("channel_id", IntegerType(), True),
    StructField("channel_name", StringType(), True),
    StructField("channel_type", StringType(), True)
])


In [0]:
# Lendo os CSVS e colocando nos dataframes
base_path = "/Volumes/dc_catalog/bronze/landingzone/"

hub_df = spark.read.schema(hubs_schema).csv(f"{base_path}/hubs.csv", header=False) # Esse tipo de exportação perde os cabeçalhos
store_df = spark.read.schema(stores_schema).csv(f"{base_path}/stores.csv", header=False)
drivers_df = spark.read.schema(drivers_schema).csv(f"{base_path}/drivers.csv", header=False)
channels_df = spark.read.schema(channels_schema).csv(f"{base_path}/channels.csv", header=False)
payments_df = spark.read.schema(payments_schema).csv(f"{base_path}/payments.csv", header=False)
deliveries_df = spark.read.schema(deliveries_schema).csv(f"{base_path}/deliveries.csv", header=False)
orders_df = spark.read.schema(orders_schema).csv(f"{base_path}/orders.csv", header=False)

### Salvando os dados na Camada Bronze

In [0]:
bronze_catalog = "dc_catalog.bronze" #Aqui, salvamos os dados em tabelas na camada Bronze 
hub_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.hubs")
store_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.stores")
drivers_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.drivers")
channels_df.write.format("delta").mode("overwrite").saveAsTable(f'{bronze_catalog}.channels')
payments_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.payments")
deliveries_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.deliveries")
orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{bronze_catalog}.orders")

print("Todas as tabelas foram salvas na camada Bronze (Delta).")


In [0]:
%sql
-- Aqui, verificamos se foi tudo correto, é apenas uma célula de teste e visua
--SELECT * FROM dc_catalog.bronze.hubs LIMIT 5;
--SELECT * FROM dc_catalog.bronze.stores LIMIT 5;
--SELECT * FROM dc_catalog.bronze.drivers LIMIT 5;
--SELECT * FROM dc_catalog.bronze.channels LIMIT 5;
--SELECT * FROM dc_catalog.bronze.payments LIMIT 5;
--SELECT * FROM dc_catalog.bronze.deliveries LIMIT 5;
SELECT * FROM dc_catalog.bronze.orders LIMIT 50;

In [0]:
%sql
-- Aqui é apenas uma célula que verifica a quantidade de nulos da maior tabela, auxiliando no tratamento dos dados
SELECT
    COUNT(CASE WHEN order_id IS NULL THEN 1 END) AS order_id_nulls,
    COUNT(CASE WHEN store_id IS NULL THEN 1 END) AS store_id_nulls,
    COUNT(CASE WHEN channel_id IS NULL THEN 1 END) AS channel_id_nulls,
    COUNT(CASE WHEN payment_order_id IS NULL THEN 1 END) AS payment_order_id_nulls,
    COUNT(CASE WHEN delivery_order_id IS NULL THEN 1 END) AS delivery_order_id_nulls,
    COUNT(CASE WHEN order_status IS NULL THEN 1 END) AS order_status_nulls,
    COUNT(CASE WHEN order_amount IS NULL THEN 1 END) AS order_amount_nulls,
    COUNT(CASE WHEN order_delivery_fee IS NULL THEN 1 END) AS order_delivery_fee_nulls,
    COUNT(CASE WHEN order_delivery_cost IS NULL THEN 1 END) AS order_delivery_cost_nulls,
    COUNT(CASE WHEN order_created_hour IS NULL THEN 1 END) AS order_created_hour_nulls,
    COUNT(CASE WHEN order_created_minute IS NULL THEN 1 END) AS order_created_minute_nulls,
    COUNT(CASE WHEN order_created_day IS NULL THEN 1 END) AS order_created_day_nulls,
    COUNT(CASE WHEN order_created_month IS NULL THEN 1 END) AS order_created_month_nulls,
    COUNT(CASE WHEN order_created_year IS NULL THEN 1 END) AS order_created_year_nulls,
    COUNT(CASE WHEN order_moment_created IS NULL THEN 1 END) AS order_moment_created_nulls,
    COUNT(CASE WHEN order_moment_accepted IS NULL THEN 1 END) AS order_moment_accepted_nulls,
    COUNT(CASE WHEN order_moment_ready IS NULL THEN 1 END) AS order_moment_ready_nulls,
    COUNT(CASE WHEN order_moment_collected IS NULL THEN 1 END) AS order_moment_collected_nulls,
    COUNT(CASE WHEN order_moment_in_expedition IS NULL THEN 1 END) AS order_moment_in_expedition_nulls,
    COUNT(CASE WHEN order_moment_delivering IS NULL THEN 1 END) AS order_moment_delivering_nulls,
    COUNT(CASE WHEN order_moment_delivered IS NULL THEN 1 END) AS order_moment_delivered_nulls,
    COUNT(CASE WHEN order_moment_finished IS NULL THEN 1 END) AS order_moment_finished_nulls,
    COUNT(CASE WHEN order_metric_collected_time IS NULL THEN 1 END) AS order_metric_collected_time_nulls,
    COUNT(CASE WHEN order_metric_paused_time IS NULL THEN 1 END) AS order_metric_paused_time_nulls,
    COUNT(CASE WHEN order_metric_production_time IS NULL THEN 1 END) AS order_metric_production_time_nulls,
    COUNT(CASE WHEN order_metric_walking_time IS NULL THEN 1 END) AS order_metric_walking_time_nulls,
    COUNT(CASE WHEN order_metric_expediton_speed_time IS NULL THEN 1 END) AS order_metric_expediton_speed_time_nulls,
    COUNT(CASE WHEN order_metric_transit_time IS NULL THEN 1 END) AS order_metric_transit_time_nulls,
    COUNT(CASE WHEN order_metric_cycle_time IS NULL THEN 1 END) AS order_metric_cycle_time_nulls
FROM dc_catalog.bronze.orders;

### Limpando a Landing Zone

In [0]:
%fs
rm -r /Volumes/dc_catalog/bronze/landingzone/*

# Camada Silver

### Leitura dos dados na Camada Bronze

In [0]:
# Caminho base da camada Bronze
bronze_path = "dc_catalog.bronze"

silver_hubs_df = (spark.read.format("delta").table(f"{bronze_path}.hubs"))
display(silver_hubs_df)

silver_stores_df = (spark.read.format("delta").table(f"{bronze_path}.stores"))
display(silver_stores_df)

silver_drivers_df = (spark.read.format("delta").table(f"{bronze_path}.drivers"))
display(silver_drivers_df)

silver_channels_df = (spark.read.format("delta").table(f"{bronze_path}.channels"))
display(silver_channels_df)

silver_payments_df = (spark.read.format("delta").table(f"{bronze_path}.payments"))
display(silver_payments_df)

silver_deliveries_df = (spark.read.format("delta").table(f"{bronze_path}.deliveries"))
display(silver_deliveries_df)

silver_orders_df = (spark.read.format("delta").table(f"{bronze_path}.orders"))
display(silver_orders_df)


### Limpeza e Padronização de Dados

In [0]:
# Camada Silver - Tabela dc_catalog.silver.orders
# Objetivo:
# - Limpar e padronizar os dados da Bronze (dc_catalog.bronze.orders)
# - Remover colunas inúteis
# - Tratar valores nulos em métricas e valores financeiros
# - Padronizar tipos e categorias
# - Remover possíveis registros duplicados

# Leitura da camada Bronze (dados brutos, sem alterações)
bronze = spark.table("dc_catalog.bronze.orders")

# Identificação das colunas 'order_moment_*' para remoção
#
# Essas colunas representam timestamps de etapas do pedido, mas
# na auditoria foram identificadas como praticamente 100% nulas.
# Como não carregam informação útil, serão removidas na Silver.
cols_drop = [c for c in bronze.columns if c.startswith("order_moment_")]

silver = (
    bronze

    # Remoção das colunas inutilizáveis (order_moment_*)
    .drop(*cols_drop)

    # Tratamento das métricas de tempo (order_metric_*)
    #
    # As métricas de tempo representam a duração, em segundos, de
    # etapas do fluxo logístico (pausa, produção, caminhada, expedição,
    # trânsito e ciclo total do pedido).
    #
    # Quando o valor é NULL, isso indica que a etapa não ocorreu
    # (ex.: pedido cancelado antes de sair para entrega, nenhuma pausa
    # registrada, etc.), e não que o dado foi perdido.
    #
    # Regra adotada:
    # - NULL nessas colunas => 0 segundos (não houve tempo gasto).
    .withColumn("order_metric_paused_time",
                F.coalesce("order_metric_paused_time", F.lit(0)))
    .withColumn("order_metric_production_time",
                F.coalesce("order_metric_production_time", F.lit(0)))
    .withColumn("order_metric_walking_time",
                F.coalesce("order_metric_walking_time", F.lit(0)))
    .withColumn("order_metric_expediton_speed_time",
            F.coalesce("order_metric_expediton_speed_time", F.lit(0)))
    .withColumn("order_metric_transit_time",
                F.coalesce("order_metric_transit_time", F.lit(0)))
    .withColumn("order_metric_cycle_time",
                F.coalesce("order_metric_cycle_time", F.lit(0)))
    
    # Tratamento de valores financeiros
    #
    # As colunas financeiras representam custos e taxas:
    # - order_delivery_cost: custo da entrega
    # - order_delivery_fee : taxa de entrega cobrada
    #
    # Quando aparecem como NULL, significa que não houve cobrança
    # (pedido cancelado antes da cobrança, pedido gratuito ou
    # taxa subsidiada). Ou seja:
    # - NULL => custo / taxa igual a 0.
    .withColumn("order_delivery_cost",
                F.coalesce("order_delivery_cost", F.lit(0)))
    .withColumn("order_delivery_fee",
                F.coalesce("order_delivery_fee", F.lit(0)))

    # Tratamento de variáveis categóricas
    #
    # Em colunas categóricas, um valor NULL representa "não informado".
    # Para evitar problemas em joins e em modelos de ML, substituímos
    # esses casos por um rótulo padrão.
    #
    # Regra adotada:
    # - NULL em order_status => "UNKNOWN"
    .withColumn("order_status",
                F.coalesce("order_status", F.lit("UNKNOWN")))
)

# Padronização de tipos de dados
# 
# Aqui garantimos que colunas numéricas estejam em tipos apropriados
# para análises e ML (double/long). Caso algum cast não seja necessário
# no seu schema, pode ser removido.
silver = (
    silver
    .withColumn("order_amount", F.col("order_amount").cast(DoubleType()))
    .withColumn("order_delivery_fee", F.col("order_delivery_fee").cast(DoubleType()))
    .withColumn("order_delivery_cost", F.col("order_delivery_cost").cast(DoubleType()))
    .withColumn("order_metric_cycle_time",
                F.col("order_metric_cycle_time").cast(LongType()))
    .withColumn("order_metric_transit_time",
                F.col("order_metric_transit_time").cast(LongType()))
)

# Normalização de valores categóricos
# 
# Padronizamos o texto das categorias para evitar variações como
# "Canceled", "CANCELED", "cancelado", etc. A ideia é que cada status
# tenha sempre a mesma grafia.
#
# Aqui usamos upper() para deixar tudo em maiúsculo e, em seguida,
# normalizamos algumas variações comuns (ex.: CANCELLED -> CANCELED).
silver = (
    silver
    # tudo em maiúsculo
    .withColumn("order_status", F.upper(F.col("order_status")))
    # normalização de variações do status de cancelamento
    .withColumn(
        "order_status",
        F.when(F.col("order_status").isin("CANCELLED", "CANCELED"),
               F.lit("CANCELED"))
         .otherwise(F.col("order_status"))
    )
)

silver_orders_df = silver

# Remoção de possíveis registros duplicados
# 
# Caso existam linhas duplicadas para o mesmo order_id, mantemos
# apenas uma ocorrência. Isso ajuda a preservar a integridade dos
# dados na Silver e evita contagens infladas na Gold.
silver = silver.dropDuplicates(["order_id"])


In [0]:
def null_profile(df, nome_df):
    """
    Gera um DataFrame com quantidade e percentual de valores nulos
    para cada coluna de um DataFrame Spark.
    """
    total = df.count()

    exprs = [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]

    null_counts = df.agg(*exprs).collect()[0].asDict()

    linhas = [
        (nome_df, col, int(null_counts[col]), float(null_counts[col]) / total)
        for col in df.columns
    ]

    return spark.createDataFrame(linhas, ["dataframe", "coluna", "qtd_nulos", "perc_nulos"])


# Lista de todos os DataFrames Silver que você está usando
silver_dfs = [
    ("silver_hubs_df",       silver_hubs_df),
    ("silver_stores_df",     silver_stores_df),
    ("silver_drivers_df",    silver_drivers_df),
    ("silver_channels_df",   silver_channels_df),
    ("silver_payments_df",   silver_payments_df),
    ("silver_deliveries_df", silver_deliveries_df),
    ("silver_orders_df",     silver_orders_df),  # já limpo
]

# Gera o perfil de nulos de todos
perfil_nulls = None

for nome, df in silver_dfs:
    perfil_df = null_profile(df, nome)
    if perfil_nulls is None:
        perfil_nulls = perfil_df
    else:
        perfil_nulls = perfil_nulls.unionByName(perfil_df)

# Se por algum motivo não tiver nada, evita erro
if perfil_nulls is None:
    print("Nenhum DataFrame Silver foi analisado.")
else:
    # Filtra somente colunas que ainda têm nulos e ordena
    perfil_nulls_restante = (
        perfil_nulls
        .filter(F.col("qtd_nulos") > 0)
        .orderBy("dataframe", F.desc("qtd_nulos"))
    )

    display(perfil_nulls_restante)

In [0]:
# Tratamento das colunas restantes com valor nulo

# silver_deliveries_df
# A coluna driver_id possui praticamente 100% dos valores nulos.
# Isso indica ausência real de informação e impossibilidade de uso em análises,
# joins ou modelos de Machine Learning. Por isso, removemos a coluna.
silver_deliveries_df = silver_deliveries_df.drop("driver_id")

# delivery_distance_meters -> poucos nulos -> substituímos por 0
silver_deliveries_df = (
    silver_deliveries_df
        .withColumn("delivery_distance_meters",
            F.coalesce("delivery_distance_meters", F.lit(0)))
)

# silver_orders_df
# order_metric_collected_time -> nulo significa não coletado -> substituímos por 0 segundos
silver_orders_df = (
    silver_orders_df
        .withColumn("order_metric_collected_time",
            F.coalesce("order_metric_collected_time", F.lit(0)))
)

# silver_payments_df
# payment_fee -> nulo = taxa não cobrada -> substituir por 0
silver_payments_df = (
    silver_payments_df
        .withColumn("payment_fee",
            F.coalesce("payment_fee", F.lit(0)))
)

# silver_stores_df
# store_plan_price -> nulo = plano não registrado -> 0
silver_stores_df = (
    silver_stores_df
        .withColumn("store_plan_price",
            F.coalesce("store_plan_price", F.lit(0)))
)

# latitude e longitude nulos -> substituímos por 0 para evitar erro no ML
silver_stores_df = (
    silver_stores_df
        .withColumn("store_latitude",
            F.coalesce("store_latitude", F.lit(0)))
        .withColumn("store_longitude",
            F.coalesce("store_longitude", F.lit(0)))
)


print("Tratamento final de nulos aplicado com sucesso!")


In [0]:
spark.sql("DROP TABLE IF EXISTS dc_catalog.silver.deliveries")
spark.sql("DROP TABLE IF EXISTS dc_catalog.silver.orders")
spark.sql("DROP TABLE IF EXISTS dc_catalog.silver.payments")
spark.sql("DROP TABLE IF EXISTS dc_catalog.silver.stores")



### Salvando os dados na Camada Silver

In [0]:
silver_catalog = "dc_catalog.silver"

silver_hubs_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.hubs")
silver_stores_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.stores")
silver_drivers_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.drivers")
silver_channels_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.channels")
silver_payments_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.payments")
silver_deliveries_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.deliveries")
silver_orders_df.write.format("delta").mode("overwrite").saveAsTable(f"{silver_catalog}.orders")

print("Todas as tabelas foram salvas na camada Silver (Delta).")


## Análise posteriores na camada silver

In [0]:
# Análise exploratória na camada Silver (usando apenas Spark)
# Tabela: dc_catalog.silver.orders

silver = spark.table("dc_catalog.silver.orders")

# Estatísticas descritivas das colunas numéricas
# 
# Aqui usamos o describe() do Spark para obter:
# - count, mean, stddev, min, max
# Isso ajuda a entender faixas de valores, dispersão e possíveis outliers.
num_cols = [
    "order_amount",
    "order_delivery_fee",
    "order_delivery_cost",
    "order_metric_cycle_time",
    "order_metric_transit_time",
    "order_metric_production_time",
]

stats_df = silver.select(num_cols).describe()
display(stats_df)

# Aproximação de quantis (distribuição) com approxQuantile
# 
# Para evitar trazer todos os dados para o driver, usamos o approxQuantile,
# que é nativo do Spark e retorna quantis aproximados.
# Isso serve para descrever a distribuição sem sair do contexto distribuído.
quantis = [0.25, 0.5, 0.75, 0.9]

for col in num_cols:
    q = silver.approxQuantile(col, quantis, 0.01)
    print(f"Quantis de {col}:")
    print(f"  25%: {q[0]}  |  50% (mediana): {q[1]}  |  75%: {q[2]}  |  90%: {q[3]}")
    print("-" * 60)

# Distribuição de status dos pedidos (variável categórica)
# 
# Aqui analisamos quantos pedidos existem em cada status.
# Isso mostra, por exemplo, proporção de pedidos cancelados, entregues etc.
status_dist_df = (
    silver
    .groupBy("order_status")
    .count()
    .orderBy(F.col("count").desc())
)

display(status_dist_df)

# Comentário sugerido para relatório:
# "A análise da distribuição de 'order_status' permite identificar o desbalanceamento
# entre pedidos entregues e cancelados, informação importante tanto para
# avaliação operacional quanto para os modelos de classificação."

# Correlação entre variáveis numéricas
# 
# O Spark permite calcular correlação entre pares de colunas numéricas
# usando o método stat.corr. Abaixo montamos uma pequena "matriz"
# de correlação só com Spark (sem Pandas).
corr_pairs = [
    ("order_amount", "order_metric_cycle_time"),
    ("order_delivery_fee", "order_metric_cycle_time"),
    ("order_metric_transit_time", "order_metric_cycle_time"),
    ("order_metric_production_time", "order_metric_cycle_time"),
]

corr_results = []
for col_x, col_y in corr_pairs:
    corr_value = silver.stat.corr(col_x, col_y)
    corr_results.append((col_x, col_y, corr_value))

# Converte a lista em DataFrame Spark para exibir organizado
corr_df = spark.createDataFrame(
    corr_results,
    ["coluna_x", "coluna_y", "correlacao_pearson"]
)

display(corr_df)

# A correlação de Pearson calculada diretamente no Spark indica o grau de relação
# linear entre as variáveis numéricas. Em especial, a correlação entre
# order_metric_transit_time e order_metric_cycle_time evidencia o impacto
# do tempo de trânsito no tempo total do pedido.

In [0]:
# Histograma de uma coluna numérica usando apenas Spark

# Número de bins (faixas)
bins = 10

# Pega os quantis que formam os limites dos bins
quantis = [i / bins for i in range(bins + 1)]
bounds = silver.approxQuantile("order_amount", quantis, 0.01)

print("Limites dos bins:")
print(bounds)

# Agora contamos quantos valores caem em cada faixa usando Spark
from pyspark.sql import functions as F

bin_counts = []
for i in range(len(bounds) - 1):
    lower = bounds[i]
    upper = bounds[i+1]

    count = silver.filter(
        (F.col("order_amount") >= lower) &
        (F.col("order_amount") < upper)
    ).count()

    bin_counts.append((f"{lower:.2f} - {upper:.2f}", count))

df_bins = spark.createDataFrame(bin_counts, ["faixa", "quantidade"])
display(df_bins)

# O histograma calculado via Spark RDD.histogram indica que a maior
# concentração de pedidos encontra-se nas faixas de menor valor de order_amount,
# com queda de frequência nas faixas superiores, caracterizando uma distribuição
# assimétrica à direita.

# Camada Gold
- Uso das tabelas **Silver existentes**
- Implementação da **Camada Gold**
- **Esquema Estrela** com `fact_orders`
- Apache Spark + Delta Lake

### Leitura dos dados na Camada Gold

In [0]:
# Caminho base da camada Silver
silver_path = "dc_catalog.silver"

gold_hubs_df = (spark.read.format("delta").table(f"{silver_path}.hubs"))
display(gold_hubs_df)

gold_stores_df = (spark.read.format("delta").table(f"{silver_path}.stores"))
display(gold_stores_df)

gold_drivers_df = (spark.read.format("delta").table(f"{silver_path}.drivers"))
display(gold_drivers_df)

gold_channels_df = (spark.read.format("delta").table(f"{silver_path}.channels"))
display(gold_channels_df)

gold_payments_df = (spark.read.format("delta").table(f"{silver_path}.payments"))
display(gold_payments_df)

gold_deliveries_df = (spark.read.format("delta").table(f"{silver_path}.deliveries"))
display(gold_deliveries_df)

gold_orders_df = (spark.read.format("delta").table(f"{silver_path}.orders"))
display(gold_orders_df)

## Tabelas de Dimensão e Fato

![Esquema Estrela](https://imgur.com/XpzrLAu.png)

### Tabela de Dimensão do Channel (`DimChannel`)

In [0]:
gold_schema = "dc_catalog.gold"
silver_schema = "dc_catalog.silver"

dim_channel_df = (
    spark.table(f"{silver_schema}.channels")
         .select("channel_id", "channel_name", "channel_type")
         .dropDuplicates()
)
dim_channel_df = dim_channel_df.withColumn( # Adicionar surrogate key
    "sk_channel", monotonically_increasing_id() + 1
)
dim_channel_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_channel") # Salvar em Delta

display(dim_channel_df)


### Tabela de Dimensão dos Payments (`DimPayment`)

In [0]:
dim_payment_df = (
    spark.table(f"{silver_schema}.payments")
         .select(
             "payment_id",
             "payment_method",
             "payment_status"
         ).dropDuplicates()
)

dim_payment_df = dim_payment_df.withColumn(
    "sk_payment", monotonically_increasing_id() + 1
)

dim_payment_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_payment")

display(dim_payment_df)

### Tabela de Dimensão dos Drivers (`DimDriver`)

In [0]:
dim_driver_df = (
    spark.table(f"{silver_schema}.drivers")
         .select("driver_id", "driver_modal", "driver_type")
         .dropDuplicates()
)

dim_driver_df = dim_driver_df.withColumn(
    "sk_driver", monotonically_increasing_id() + 1
)

dim_driver_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_driver")

display(dim_driver_df)


### Tabela de Dimensão do Delivery (`DimDelivery`)

In [0]:
dim_delivery_df = (
    spark.table(f"{silver_schema}.deliveries")
         .select(
             "delivery_id",
             "delivery_order_id",
             "delivery_distance_meters",
             "delivery_status"
         )
         .dropDuplicates()
)

dim_delivery_df = dim_delivery_df.withColumn(
    "dim_delivery_key", monotonically_increasing_id() + 1
)

dim_delivery_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_delivery")

display(dim_delivery_df)

### Tabela de Dimensão do Hub (`DimHub`)

In [0]:
dim_hub_df = (
    spark.table(f"{silver_schema}.hubs")
         .select(
             "hub_id",
             "hub_name",
             "hub_city",
             "hub_state",
             "hub_latitude",
             "hub_longitude"
         )
         .dropDuplicates()
)

dim_hub_df = dim_hub_df.withColumn(
    "sk_hub", monotonically_increasing_id() + 1
)

dim_hub_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_hub")

display(dim_hub_df)


### Tabela de Dimensão do Stores (`DimStore`)

In [0]:
dim_store_df = (
    spark.table(f"{silver_schema}.stores")
         .select(
             "store_id",
             "hub_id",
             "store_name",
             "store_segment",
             "store_plan_price",
             "store_latitude",
             "store_longitude"
         )
         .dropDuplicates()
)

dim_store_df = dim_store_df.withColumn(
    "sk_store", monotonically_increasing_id() + 1
)

dim_store_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{gold_schema}.dim_store")

display(dim_store_df)


### Tabela de Fato dos Pedidos (`FactOrders`)
- Para carregar os dados nessa tabela, além dos dados da tabela orders, também vai ser necessário obter os dados das chaves das outras tabelas
- Para isso, é necessário realizar uma junção da tabela orders com as tabelas de dimensão

In [0]:

silver = "dc_catalog.silver"
gold = "dc_catalog.gold"

orders_df = spark.table(f"{silver}.orders")
dim_store     = spark.table(f"{gold}.dim_store")
dim_channel   = spark.table(f"{gold}.dim_channel")
dim_payment   = spark.table(f"{gold}.dim_payment")
dim_delivery  = spark.table(f"{gold}.dim_delivery")
dim_hub       = spark.table(f"{gold}.dim_hub")

fact_orders = (
    orders_df
    # STORE → SK
    .join(dim_store.select("store_id", "sk_store"), on="store_id", how="left")
    # CHANNEL → SK
    .join(dim_channel.select("channel_id", "sk_channel"), on="channel_id", how="left")
    # PAYMENT → SK
    .join(dim_payment.select("payment_id", "sk_payment"),
          orders_df.payment_order_id == dim_payment.payment_id, how="left")
    # DELIVERY → SK
    .join(dim_delivery.select("delivery_order_id", "dim_delivery_key"),
          orders_df.delivery_order_id == dim_delivery.delivery_order_id, how="left")
    # HUB (via loja)
    .join(dim_hub.select("hub_id", "sk_hub"),
          dim_store.hub_id == dim_hub.hub_id, how="left")
)

fact_orders = fact_orders.selectExpr( #Foi usada a Expr por causa do tipo das duas ultimas colunas, para permitir a expressão que as deixa em bigint
    # --- Surrogate Keys ---
    "sk_store",
    "sk_channel",
    "sk_payment",
    "dim_delivery_key",
    "sk_hub",
    # --- Business Keys ---
    "order_id",
    # --- Métricas ---
    "order_amount",
    "order_delivery_fee",
    "order_delivery_cost",
    # --- Datas / horários ---
    "order_created_hour",
    "order_created_minute",
    "order_created_day",
    "order_created_month",
    "order_created_year",
    # --- Métricas de tempo ---
    "order_metric_collected_time",
    "order_metric_paused_time",
    "order_metric_production_time",
    "order_metric_walking_time",
    "order_metric_expediton_speed_time",
    "CAST(order_metric_transit_time AS BIGINT) AS order_metric_transit_time",
    "CAST(order_metric_cycle_time AS BIGINT) AS order_metric_cycle_time"
)

fact_orders.write.format("delta").mode("overwrite").saveAsTable(f"{gold}.fact_orders")

display(fact_orders)


## Salvando dados na Camada Gold

In [0]:
# Catálogo/schema da Camada Gold
gold_catalog = "dc_catalog.gold"

# Salvando as tabelas da Gold
gold_hubs_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.hubs")
gold_stores_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.stores")
gold_drivers_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.drivers")
gold_channels_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.channels")
gold_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.payments")
gold_deliveries_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.deliveries")
gold_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_catalog}.orders")

# Ler a tabela de orders da Gold (para a MLP)
gold_orders_spark = spark.table(f"{gold_catalog}.orders")

print("Todas as tabelas foram salvas na camada Gold (Delta).")


# Aplicação das Redes Neurais

## MLP de Classificação

In [0]:
# REDE NEURAL DE CLASSIFICAÇÃO (MLPClassifier)
# Alvo: label_cancelado (1 = CANCELED, 0 = outros status)
# Fonte: dc_catalog.gold.orders

# Ler a Gold no Spark
gold_orders_spark = spark.table("dc_catalog.gold.orders")

# Criar a coluna alvo binária
gold_orders_spark = gold_orders_spark.withColumn(
    "label_cancelado",
    F.when(F.col("order_status") == "CANCELED", 1).otherwise(0)
)

# Definir colunas numéricas e categóricas
cols_numericas = [
    "order_amount",
    "order_delivery_fee",
    "order_delivery_cost",
    "order_metric_cycle_time",
    "order_metric_transit_time",
    "order_metric_production_time",
    "delivery_distance_meters",
    "store_plan_price"
]

cols_categoricas = [
    "store_city",
    "store_segment",
    "channel_name"
]

# Garante que só usa colunas que realmente existem
cols_features = [c for c in cols_numericas + cols_categoricas if c in gold_orders_spark.columns]
print("Features selecionadas (classificação):", cols_features)

df_spark_clf = gold_orders_spark.select(*cols_features, "label_cancelado").dropna()

# Converter Spark para Pandas
df_clf = df_spark_clf.toPandas()
print("Shape em Pandas (classificação):", df_clf.shape)

# Separar X e y
y_clf = df_clf["label_cancelado"]
X_clf = df_clf.drop(columns=["label_cancelado"])

# One-hot encoding das categóricas
X_clf = pd.get_dummies(
    X_clf,
    columns=[c for c in cols_categoricas if c in X_clf.columns],
    drop_first=True
)

print("Shape de X_clf após dummies:", X_clf.shape)

# Train/Test split usando padrão 80%/20%
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

# Normalização
scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled  = scaler_clf.transform(X_test_clf)

# Definir e treinar a DNN de classificação (MLPClassifier)
mlp_clf = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate='adaptive',
    max_iter=200,
    random_state=42,
    verbose=False
)

mlp_clf.fit(X_train_clf_scaled, y_train_clf)

# Avaliação
y_pred_clf = mlp_clf.predict(X_test_clf_scaled)

print("\nAcurácia no conjunto de teste (classificação):")
print((y_pred_clf == y_test_clf).mean())

print("\nRelatório de Classificação:")
print(classification_report(y_test_clf, y_pred_clf))

cm_clf = confusion_matrix(y_test_clf, y_pred_clf)
print("Matriz de confusão:\n", cm_clf)

# Heatmap da matriz de confusão
plt.figure(figsize=(6,5))
sns.heatmap(cm_clf, annot=True, fmt='d', cmap='Purples')
plt.xlabel('Classe Prevista')
plt.ylabel('Classe Real')
plt.title('Matriz de Confusão - Cancelamento de Pedido (MLPClassifier)')
plt.tight_layout()
plt.show()


## DNN de Regressão

In [0]:
# REDE NEURAL DE REGRESSÃO (MLPRegressor)
# Alvo: order_metric_cycle_time (tempo de ciclo do pedido)
# Fonte: dc_catalog.gold.orders

# Ler a Gold no Spark
gold_orders_spark = spark.table("dc_catalog.gold.orders")

# Definir coluna alvo e features
col_alvo_reg = "order_metric_cycle_time"

cols_numericas_reg = [
    "order_amount",
    "order_delivery_fee",
    "order_delivery_cost",
    "order_metric_transit_time",
    "order_metric_production_time",
    "delivery_distance_meters",
    "store_plan_price"
]

cols_categoricas_reg = [
    "store_city",
    "store_segment",
    "channel_name",
    "order_status"
]

cols_features_reg = [c for c in cols_numericas_reg + cols_categoricas_reg
                     if c in gold_orders_spark.columns]

print("Features selecionadas (regressão):", cols_features_reg)

df_spark_reg = gold_orders_spark.select(*cols_features_reg, col_alvo_reg).dropna()

# Converter Spark para Pandas
df_reg = df_spark_reg.toPandas()
print("Shape em Pandas (regressão):", df_reg.shape)

# Separar X e y
y_reg = df_reg[col_alvo_reg]
X_reg = df_reg.drop(columns=[col_alvo_reg])

# One-hot encoding
X_reg = pd.get_dummies(
    X_reg,
    columns=[c for c in cols_categoricas_reg if c in X_reg.columns],
    drop_first=True
)

print("Shape de X_reg após dummies:", X_reg.shape)

# Train/Test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

# Normalização
scaler_reg = StandardScaler()
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled  = scaler_reg.transform(X_test_reg)

# Definir e treinar a DNN de regressão (MLPRegressor)
mlp_reg = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate='adaptive',
    max_iter=300,
    random_state=42,
    verbose=False
)

mlp_reg.fit(X_train_reg_scaled, y_train_reg)

# Predição e métricas
y_pred_reg = mlp_reg.predict(X_test_reg_scaled)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"\nMAE  (erro médio absoluto): {mae:.4f}")
print(f"RMSE (raiz do erro quadrático médio): {rmse:.4f}")
print(f"R²   (coeficiente de determinação): {r2:.4f}")

#   Gráfico Real vs Previsto
plt.figure(figsize=(6,6))
plt.scatter(y_test_reg, y_pred_reg, alpha=0.4)
plt.xlabel('Valor Real')
plt.ylabel('Valor Previsto')
plt.title(f'Real vs Previsto - {col_alvo_reg} (MLPRegressor)')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


# Análises posteriores

In [0]:
# Função auxiliar: preparar features novas (a partir da Gold) para o modelo já treinado

# Garante que temos a lista de colunas de features usada no treino
feature_cols_clf = list(X_clf.columns)

def preparar_features_para_modelo(df_spark, cols_numericas, cols_categoricas):
    # Recebe um DataFrame Spark com as colunas necessárias, devolve:
    #
    # - df_original (Pandas) com todas as colunas originais
    # - X_scaled (numpy) com as features alinhadas e normalizadas,
    # pronto para passar no model_clf.predict

    # Converte para Pandas
    df = df_spark.toPandas()
    
    # Aplica one-hot nas colunas categóricas (mesmo padrão do treino)
    df_encoded = pd.get_dummies(df, columns=cols_categoricas, drop_first=True)
    
    # Garante que TODAS as colunas usadas no treino existem aqui
    for col in feature_cols_clf:
        if col not in df_encoded.columns:
            # coluna ausente para o modelo, preenche com 0
            df_encoded[col] = 0
    
    # Reordena colunas na mesma ordem do treino
    df_encoded = df_encoded[feature_cols_clf]
    
    # Aplica o scaler já fitado
    X_scaled = scaler_clf.transform(df_encoded)
    
    return df, X_scaled


## Análise 1 - Fatores que Mais Contribuem para Cancelamentos (Classificação)

In [0]:
# Predição base no conjunto de teste
y_pred_base = mlp_clf.predict(X_test_clf_scaled)
acc_base = accuracy_score(y_test_clf, y_pred_base)

print(f"Acurácia base no teste (sem permutação): {acc_base:.4f}")

# Permutation importance simples
feature_names = list(X_clf.columns)
X_test_array = X_test_clf_scaled.copy()

rng = np.random.default_rng(42)
importancias = []

for j, feat in enumerate(feature_names):
    # copia os dados de teste
    X_permutado = X_test_array.copy()

    # embaralha SOMENTE a coluna j
    rng.shuffle(X_permutado[:, j])

    # recalcula as predições e a acurácia com a coluna embaralhada
    y_pred_perm = mlp_clf.predict(X_permutado)
    acc_perm = accuracy_score(y_test_clf, y_pred_perm)

    queda = acc_base - acc_perm
    importancias.append((feat, queda))

# Organiza em DataFrame
importancias_df = pd.DataFrame(importancias,
                               columns=["feature", "queda_acuracia"])
importancias_df = importancias_df.sort_values(
    "queda_acuracia", ascending=False
)

print("\nImportância (Permutation) – Queda de acurácia ao embaralhar a feature:")
display(importancias_df.head(15))

# A análise de importância via permutação indica que variáveis como
# order_metric_transit_time, delivery_distance_meters e order_amount
# são as que mais impactam a capacidade do modelo em prever cancelamentos,
# sugerindo maior sensibilidade a tempo de trânsito, distância e valor do pedido.

## Análise 2 - Probabilidade de Cancelamento em Diferentes Perfis de Loja (Aplicação Direta do Modelo)

In [0]:
# Seleciona colunas relevantes na Gold
cols_para_analise = cols_numericas + cols_categoricas

# Garante que só usa colunas que realmente existem na tabela Gold
cols_para_analise = [c for c in cols_para_analise if c in gold_orders_spark.columns]
print("Colunas usadas na Análise 2:", cols_para_analise)

df_perf_spark = gold_orders_spark.select(*cols_para_analise)

# Converte para Pandas para aplicar o mesmo pipeline da MLP
df_perf = df_perf_spark.toPandas().copy()

# Tratamento simples de nulos (igual espírito da Silver/Gold)

# Preenche nulos numéricos com 0
for c in cols_numericas:
    if c in df_perf.columns:
        df_perf[c] = df_perf[c].fillna(0)

# Preenche nulos categóricos com 'UNKNOWN'
for c in cols_categoricas:
    if c in df_perf.columns:
        df_perf[c] = df_perf[c].fillna("UNKNOWN")

# Prepara X da mesma forma que na rede de classificação

# Colunas categóricas que realmente existem neste DF
cat_cols_exist = [c for c in cols_categoricas if c in df_perf.columns]

# Monta o X bruto (numéricas + categóricas presentes)
X_perf = df_perf[cols_para_analise].copy()

# One-hot encoding nas categóricas existentes
X_perf_dummies = pd.get_dummies(
    X_perf,
    columns=cat_cols_exist,
    drop_first=True
)

# Garante que as colunas fiquem alinhadas com as usadas no treinamento
# (X_clf vem do bloco da MLP de classificação)
X_perf_dummies = X_perf_dummies.reindex(columns=X_clf.columns, fill_value=0)

# Normaliza com o mesmo scaler usado no treino
X_perf_scaled = scaler_clf.transform(X_perf_dummies)

# Probabilidade de cancelamento pelo modelo treinado

prob_cancel = mlp_clf.predict_proba(X_perf_scaled)[:, 1]
df_perf["prob_cancelamento"] = prob_cancel

# Cria perfis de loja por faixas (exemplo: valor do pedido e "distância/tempo")

# Faixas de valor do pedido
df_perf["faixa_valor"] = pd.cut(
    df_perf["order_amount"],
    bins=[0, 60, 80, 120, 999999],
    labels=["até 60", "60-80", "80-120", "acima de 120"]
)

# Escolhe qual coluna usar como "distância/tempo" dependendo do que existe
if "delivery_distance_meters" in df_perf.columns:
    # usa distância em metros se existir
    col_dist = "delivery_distance_meters"
    bins_dist = [0, 1000, 3000, 8000, 999999]
    labels_dist = ["até 1km", "1-3km", "3-8km", "acima de 8km"]
else:
    # caso não exista na Gold, usa tempo de trânsito como proxy
    col_dist = "order_metric_transit_time"
    # 0-30min, 30-60min, 1-2h, >2h
    bins_dist = [0, 1800, 3600, 7200, 9999999]
    labels_dist = ["até 30min", "30-60min", "1-2h", "acima de 2h"]

df_perf["faixa_distancia"] = pd.cut(
    df_perf[col_dist],
    bins=bins_dist,
    labels=labels_dist
)

# Probabilidade média por perfil

resumo_perf = (
    df_perf
        .groupby(["faixa_valor", "faixa_distancia"])
        .agg(
            prob_media_cancelamento=("prob_cancelamento", "mean"),
            qtd_pedidos=("prob_cancelamento", "size")
        )
        .reset_index()
        .sort_values("prob_media_cancelamento", ascending=False)
)

print("Probabilidade média de cancelamento por perfil (valor x distância/tempo):")
display(resumo_perf)

# Utilizando o modelo de classificação, foram simulados perfis de pedidos
# a partir de faixas de valor e distância/tempo de entrega. Observou-se que
# perfis com valor baixo e grande distância/tempo apresentam maior
# probabilidade média de cancelamento, indicando perfis de risco operacional.


## Análise 3 - Predição Individual: Criar um Simulador de Cancelamento com Inputs Manuais

In [0]:
# Exemplo de pedido
novo_pedido = {
    # Numéricas
    "order_amount": 75.0,
    "order_delivery_fee": 6.0,
    "order_delivery_cost": 10.0,
    "order_metric_cycle_time": 2400.0,
    "order_metric_transit_time": 1800.0,
    "order_metric_production_time": 600.0,
    "delivery_distance_meters": 3500.0,
    "store_plan_price": 0.0,

    # Categóricas
    "store_city": "RIO DE JANEIRO",
    "store_segment": "FAST_FOOD",
    "channel_name": "MARKETPLACE",
}

# Cria DataFrame Pandas com um único pedido
df_novo = pd.DataFrame([novo_pedido])

# Tratamento simples de nulos (mesma ideia da Análise 2)
for c in cols_numericas:
    if c in df_novo.columns:
        df_novo[c] = df_novo[c].fillna(0)

for c in cols_categoricas:
    if c in df_novo.columns:
        df_novo[c] = df_novo[c].fillna("UNKNOWN")

# Prepara X do novo pedido igual ao X_clf usado no treino

# Colunas que de fato existem neste exemplo
cols_existentes = [c for c in cols_numericas + cols_categoricas if c in df_novo.columns]

X_novo = df_novo[cols_existentes].copy()

# Aplica one-hot nas categóricas existentes
cat_cols_exist = [c for c in cols_categoricas if c in X_novo.columns]

X_novo_dummies = pd.get_dummies(
    X_novo,
    columns=cat_cols_exist,
    drop_first=True
)

# Alinha as colunas com as usadas no treino da MLP (X_clf)
X_novo_dummies = X_novo_dummies.reindex(columns=X_clf.columns, fill_value=0)

# Normaliza com o mesmo scaler da classificação
X_novo_scaled = scaler_clf.transform(X_novo_dummies)

# Predição pelo modelo treinado (MLPClassifier)
prob_novo = float(mlp_clf.predict_proba(X_novo_scaled)[0][1])
classe_prevista = int(prob_novo >= 0.5)  # 1 = cancela, 0 = não cancela

print("Dados do pedido simulado:")
display(df_novo)

print(f"\nProbabilidade prevista de cancelamento: {prob_novo:.4f}")
print(f"Classe prevista (0 = não cancela, 1 = cancela): {classe_prevista}")

# Foi implementado um simulador de pedidos, no qual o analista pode informar
# manualmente os atributos do novo pedido (valor, distância, canal, etc.)
# e obter a probabilidade de cancelamento prevista pela MLP, permitindo
# o uso interativo do modelo em cenários hipotéticos.


## Análise 4 - Curva de Risco por Valor do Pedido

In [0]:
print("Iniciando análise de risco por valor do pedido...")

# Verifica se temos as estruturas necessárias
objs_faltando = []
for nome in ["df_clf", "X_clf", "scaler_clf", "mlp_clf"]:
    if nome not in globals():
        objs_faltando.append(nome)

if objs_faltando:
    print("Não encontrei os objetos:", objs_faltando)
    print("Execute antes o bloco da MLP de classificação para criar df_clf, X_clf, scaler_clf e mlp_clf.")
else:
    # Garante que a coluna order_amount existe no df_clf
    if "order_amount" not in df_clf.columns:
        raise KeyError("A coluna 'order_amount' não está presente em df_clf. Verifique o pipeline de features.")

    # Calcula a probabilidade de cancelamento para TODO o dataset
    #    (usando as mesmas features X_clf do treino)
    X_all_scaled = scaler_clf.transform(X_clf)
    prob_all = mlp_clf.predict_proba(X_all_scaled)[:, 1]

    df_analise_valor = df_clf.copy()
    df_analise_valor["prob_cancelamento"] = prob_all

    # Remove linhas com order_amount nulo, se existirem
    df_analise_valor = df_analise_valor.dropna(subset=["order_amount"])

    # Cria faixas de valor do pedido
    df_analise_valor["faixa_valor"] = pd.cut(
        df_analise_valor["order_amount"],
        bins=[0, 20, 40, 60, 80, 100, 150, 300, 10000],
        labels=[
            "0–20",
            "20–40",
            "40–60",
            "60–80",
            "80–100",
            "100–150",
            "150–300",
            ">300"
        ],
        include_lowest=True
    )

    # Agrega probabilidade média por faixa de valor
    resumo_valor = (
        df_analise_valor
            .groupby("faixa_valor")
            .agg(
                prob_media=("prob_cancelamento", "mean"),
                qtd_pedidos=("prob_cancelamento", "size")
            )
            .reset_index()
            .dropna()
    )

    print("Probabilidade média de cancelamento por faixa de valor do pedido:")
    display(resumo_valor)

    # Gráfico — Curva de risco por faixa de valor
    plt.figure(figsize=(10, 6))
    plt.plot(
        resumo_valor["faixa_valor"].astype(str),
        resumo_valor["prob_media"],
        marker="o",
        linewidth=2
    )
    plt.title("Curva de Risco de Cancelamento por Valor do Pedido")
    plt.xlabel("Faixa de valor do pedido (order_amount)")
    plt.ylabel("Probabilidade média de cancelamento")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Gráfico de barras para reforçar
    plt.figure(figsize=(10, 6))
    plt.bar(
        resumo_valor["faixa_valor"].astype(str),
        resumo_valor["prob_media"]
    )
    plt.title("Probabilidade Média de Cancelamento por Faixa de Valor")
    plt.xlabel("Faixa de valor do pedido (order_amount)")
    plt.ylabel("Probabilidade média de cancelamento")
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Observa-se que a probabilidade de cancelamento não é constante ao longo do valor do pedido.
    #  Certas faixas de preço apresentam maior risco previsto pelo modelo,
    #  indicando perfis de pedidos potencialmente mais sensíveis a problemas
    #  de entrega, atraso ou percepção de custo pelo cliente.


## Análise 5 - Sensibilidade ao Tempo de Entrega

In [0]:
# Seleciona somente colunas necessárias
df_sens_spark = gold_orders_spark.select(*cols_features)
df_sens = df_sens_spark.toPandas().fillna(0)

# One-hot nas categóricas
df_sens = pd.get_dummies(
    df_sens,
    columns=[c for c in cols_categoricas if c in df_sens.columns],
    drop_first=True
)

# Alinha com X_clf
df_sens = df_sens.reindex(columns=X_clf.columns, fill_value=0)

# Converte para matriz
X_base = df_sens.values

# Normaliza
X_base_scaled = scaler_clf.transform(X_base)

# Número de exemplos para simulação
N = 2000
df_sample = df_sens.sample(N, random_state=42).copy()
X_sample = scaler_clf.transform(df_sample.values)

# Coluna alvo para alterar
col_tempo = "order_metric_transit_time"

if col_tempo not in df_sample.columns:
    print(f"Coluna '{col_tempo}' não existe. Usando 'order_metric_cycle_time' como alternativa.")
    col_tempo = "order_metric_cycle_time"

idx_col = list(df_sample.columns).index(col_tempo)

# Faixas de alteração do tempo (em minutos)
alteracoes = [-30, -15, 0, 15, 30, 60, 120]

resultados = []

for delta in alteracoes:
    X_mod = df_sample.copy()
    X_mod[col_tempo] = X_mod[col_tempo] + delta
    X_mod = scaler_clf.transform(X_mod)

    probs = mlp_clf.predict_proba(X_mod)[:, 1]
    resultados.append({
        "alteracao": delta,
        "prob_media": probs.mean()
    })

df_res = pd.DataFrame(resultados)

# Gráfico do resultado
plt.figure(figsize=(10, 6))
plt.plot(df_res["alteracao"], df_res["prob_media"], marker="o", color="purple")
plt.axhline(df_res[df_res["alteracao"] == 0]["prob_media"].item(),
            color="black",
            linestyle="--",
            label="Risco original")
plt.title("Sensibilidade do Modelo ao Tempo de Entrega")
plt.xlabel("Alteração no tempo de entrega (minutos)")
plt.ylabel("Probabilidade média de cancelamento")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Resultados:")
display(df_res)

# Nesta análise de sensibilidade, avaliamos como o modelo de classificação reage a alterações
# artificiais no tempo de entrega. A partir de um subconjunto amostrado da base, incrementos e
# reduções controladas foram aplicados ao tempo estimado de trânsito, e a probabilidade média de
# cancelamento prevista pelo modelo foi calculada para cada cenário.
#
# Os resultados revelam a relação direta entre aumento de tempo de entrega e maior risco previsto
# de cancelamento, indicando que o modelo é sensível a variações logísticas e que atrasos podem
# elevar significativamente o risco operacional. Essa abordagem permite identificar limites críticos
# de desempenho e apoia decisões estratégicas relacionadas à roteirização, SLA e otimização da
# cadeia de entregas.
